# Projet Big Data - Détection d'intrusion IoT (Edge-IIoTset)

Classification du type d'attaque réseau sur le dataset Edge-IIoTset.
Cible : `Attack_type` (15 classes : trafic normal + 14 attaques).

### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay


### Chargement des données

In [ ]:
df = pd.read_csv("ML-EdgeIIoT-dataset.csv", low_memory=False)
print(df.shape)
df.head()


### Exploration

In [ ]:
# types et infos générales
df.info()


In [ ]:
# valeurs manquantes et doublons
print("valeurs manquantes :", df.isna().sum().sum())
print("doublons :", df.duplicated().sum())


In [ ]:
# combien de lignes par type d'attaque
df["Attack_type"].value_counts()


In [ ]:
# on visualise la répartition
plt.figure(figsize=(10,4))
df["Attack_type"].value_counts().plot(kind="bar")
plt.title("Répartition des classes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Nettoyage

On enlève les colonnes qui servent d'identifiants ou qui peuvent fausser le modèle
(adresses IP/MAC, payloads, URI...), puis on retire les lignes vides et les doublons.

In [ ]:
cols_a_supprimer = ["frame.time","ip.src_host","ip.dst_host","arp.src.proto_ipv4",
    "arp.dst.proto_ipv4","http.file_data","http.request.full_uri",
    "icmp.transmit_timestamp","http.request.uri.query","tcp.options",
    "tcp.payload","tcp.srcport","tcp.dstport","udp.port","mqtt.msg"]

df = df.drop(columns=cols_a_supprimer, errors="ignore")
df = df.dropna()
df = df.drop_duplicates()
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("après nettoyage :", df.shape)


In [ ]:
# corrélation entre les variables numériques
plt.figure(figsize=(10,8))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()


### Préparation X / y

In [ ]:
# la cible, convertie en nombres
le = LabelEncoder()
y = le.fit_transform(df["Attack_type"].astype(str))
classes = le.classes_

# les features : on enlève les deux colonnes cibles pour éviter la triche
X = df.drop(columns=["Attack_label","Attack_type"])

# on transforme les colonnes texte en colonnes numériques
cat = [c for c in X.columns if X[c].dtype == object]
X = pd.get_dummies(X, columns=cat)
print(X.shape)


### Séparation train / test (60 / 40)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

print("train :", X_train.shape)
print("test  :", X_test.shape)


### Entraînement des modèles

In [ ]:
modeles = {
    "Regression Logistique": LogisticRegression(max_iter=1000),
    "Arbre de decision": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

for nom, m in modeles.items():
    m.fit(X_train, y_train)
    print(nom, "ok")


### Évaluation

In [ ]:
resultats = []
for nom, m in modeles.items():
    pred = m.predict(X_test)
    resultats.append({
        "Modele": nom,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1": f1_score(y_test, pred, average="weighted", zero_division=0),
    })

pd.DataFrame(resultats).set_index("Modele").round(4)


In [ ]:
# rapport détaillé du meilleur modèle
best = RandomForestClassifier(n_estimators=100, random_state=42)
best.fit(X_train, y_train)
pred = best.predict(X_test)

print(classification_report(y_test, pred, target_names=[str(c) for c in classes], zero_division=0))


In [ ]:
# matrice de confusion
fig, ax = plt.subplots(figsize=(9,8))
ConfusionMatrixDisplay.from_predictions(y_test, pred,
    display_labels=[str(c) for c in classes], xticks_rotation=90, ax=ax, colorbar=False)
plt.title("Matrice de confusion - Random Forest")
plt.tight_layout()
plt.show()


### Conclusion

A remplir : le Random Forest donne les meilleurs résultats. La regression logistique
est trop simple (données non linéaires), l'arbre de decision surapprend. Les attaques
web (Password, SQL injection, Uploading, XSS) et Fingerprinting sont les plus dures
à reconnaitre.

Prochaine etape : refaire tourner le pipeline sur Spark (W3).